In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

In [2]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(16 * 16 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [3]:
CLASS_NAMES = {
    1: '01_palm',
    2: '05_thumb',
    3: '10_down'
}

In [4]:
model = CNN()
model.load_state_dict(torch.load('hand_gesture_cnn_99.pth'))
model.eval()
print("Model loaded successfully!")

Model loaded successfully!


In [5]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [6]:
def predict_single_image(image_path):
    img = Image.open(image_path).convert('RGB')

    img_tensor = transform(img).unsqueeze(0)

    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)

        confidance, predicted_class_idx = torch.max(probabilities, 1)

        class_id = predicted_class_idx.item()

        confidance_score = confidance.item() * 100
        predicted_gesture = CLASS_NAMES.get(class_id, "Unknown")

        print(f"---Predicted Result----")
        print(f"Predicted Class ID: {class_id}")
        print(f"Gesture name: {predicted_gesture}")
        print(f"Confidance score: {confidance_score:.2f}%")

In [12]:
my_photo_path = r"C:\Users\TAQICOMPUTERS\Desktop\cv_data\test_images\IMG_20260820_122814_359.jpg"

predict_single_image(my_photo_path)

---Predicted Result----
Predicted Class ID: 1
Gesture name: 01_palm
Confidance score: 97.46%
